# 🧬 Promoter Activity Prediction Model

Predicts promoter strength across **6 experimental conditions** from 170bp plant DNA sequences (STARR-seq dataset: Arabidopsis, Maize, Sorghum).

### Pipeline
1. Upload the dataset Excel file
2. Extract sequence features (one-hot, k-mers, GC windows)
3. Train Ridge regression models (one per condition)
4. Run in-silico mutagenesis to find active regions
5. Visualize results with interactive plots

### 6 Conditions (STARR-seq)
| # | Condition |
|---|---|
| 1 | No enhancer, dark, tobacco leaves |
| 2 | No enhancer, dark, maize protoplasts |
| 3 | With enhancer, dark, tobacco leaves |
| 4 | With enhancer, dark, maize protoplasts |
| 5 | No enhancer, light, tobacco leaves |
| 6 | With enhancer, light, tobacco leaves |

## Cell 1 — Install dependencies

In [ ]:
# All packages below are pre-installed in Colab; this cell just confirms versions
import importlib, subprocess, sys

required = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'seaborn', 'openpyxl']
missing  = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)

import numpy as np
import pandas as pd
import matplotlib
import sklearn
print(f'numpy  {np.__version__}')
print(f'pandas {pd.__version__}')
print(f'sklearn {sklearn.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print('All dependencies ready ✓')

## Cell 2 — Upload dataset

Run this cell, then click **Choose Files** and upload `capstone_dataset.xlsx`.

In [ ]:
from google.colab import files
import io, os

print('Please upload capstone_dataset.xlsx ...')
uploaded = files.upload()

# Accept whatever filename was uploaded
XLSX_PATH = next(iter(uploaded))
print(f'Uploaded: {XLSX_PATH}  ({os.path.getsize(XLSX_PATH)/1e6:.1f} MB)')

## Cell 3 — Load and merge Supplementary Tables 1 & 2

In [ ]:
import openpyxl
import warnings
warnings.filterwarnings('ignore')

print('Reading Excel file (this takes ~30s for a large workbook)...')
wb = openpyxl.load_workbook(XLSX_PATH, read_only=True, data_only=True)
print(f'Sheets found: {wb.sheetnames}')

# ---------- Table 1: sequences ----------
ws1 = wb['Supplementary Table 1']
rows1 = list(ws1.iter_rows(values_only=True))[4:]   # skip 3-row header
seq_dict = {r[0]: r[11] for r in rows1 if r[0] and r[11]}
print(f'Table 1 — sequences loaded: {len(seq_dict):,}')

# ---------- Table 2: expression ----------
ws2 = wb['Supplementary Table 2']
rows2 = list(ws2.iter_rows(values_only=True))[4:]
exp_dict = {r[0]: {'species': r[1], 'expr': r[2:8]} for r in rows2 if r[0]}
print(f'Table 2 — expression records loaded: {len(exp_dict):,}')

# ---------- Merge ----------
COND_COLS = [
    'no_enhancer_dark_tobacco',
    'no_enhancer_dark_maize',
    'with_enhancer_dark_tobacco',
    'with_enhancer_dark_maize',
    'no_enhancer_light_tobacco',
    'with_enhancer_light_tobacco',
]

COND_LABELS = [
    'No enh, Dark, Tobacco',
    'No enh, Dark, Maize',
    'With enh, Dark, Tobacco',
    'With enh, Dark, Maize',
    'No enh, Light, Tobacco',
    'With enh, Light, Tobacco',
]

records = []
for gene in set(seq_dict) & set(exp_dict):
    vals = exp_dict[gene]['expr']
    if all(v is not None and v != '#N/A' for v in vals):
        row = {'gene': gene, 'species': exp_dict[gene]['species'], 'sequence': seq_dict[gene]}
        for col, v in zip(COND_COLS, vals):
            row[col] = float(v)
        records.append(row)

df = pd.DataFrame(records)
print(f'\nMerged dataset: {len(df):,} promoters with complete data')
print(df['species'].value_counts().to_string())
df.head(3)

## Cell 4 — Feature extraction

In [ ]:
from itertools import product as iproduct
import time

SEQ_LEN = 170

def one_hot_encode(seq):
    """4 × 170 one-hot matrix → 680-dim vector"""
    mapping = {'A': 0, 'T': 1, 'C': 2, 'G': 3}
    mat = np.zeros((4, SEQ_LEN), dtype=np.float32)
    for i, nt in enumerate(seq[:SEQ_LEN]):
        if nt in mapping:
            mat[mapping[nt], i] = 1.0
    return mat.flatten()

def kmer_features(seq, k=4):
    """Normalised k-mer frequency vector"""
    kmers   = [''.join(p) for p in iproduct('ATCG', repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    vec = np.zeros(len(kmers), dtype=np.float32)
    for i in range(len(seq) - k + 1):
        km = seq[i:i+k]
        if km in kmer_idx:
            vec[kmer_idx[km]] += 1
    total = vec.sum()
    if total > 0:
        vec /= total
    return vec

def dinucleotide_position_features(seq):
    """Positional dinucleotide composition in 17 × 10bp windows → 272-dim"""
    dinucs  = [''.join([a, b]) for a in 'ATCG' for b in 'ATCG']
    dn_idx  = {dn: i for i, dn in enumerate(dinucs)}
    n_bins, bin_size = 17, SEQ_LEN // 17
    vec = np.zeros(n_bins * 16, dtype=np.float32)
    for b in range(n_bins):
        s, e = b * bin_size, min((b+1) * bin_size, SEQ_LEN - 1)
        for i in range(s, e):
            dn = seq[i:i+2]
            if dn in dn_idx:
                vec[b*16 + dn_idx[dn]] += 1
        count = e - s
        if count > 0:
            vec[b*16:(b+1)*16] /= count
    return vec

def gc_windowed(seq, window=10):
    """GC% in each 10bp sliding window → 161-dim"""
    return np.array(
        [sum(1 for c in seq[i:i+window] if c in 'GC') / window
         for i in range(SEQ_LEN - window + 1)],
        dtype=np.float32
    )

def extract_features(seq):
    """Full 1,433-dim feature vector"""
    return np.concatenate([
        one_hot_encode(seq),              # 680
        kmer_features(seq, k=3),          # 64
        kmer_features(seq, k=4),          # 256
        dinucleotide_position_features(seq),  # 272
        gc_windowed(seq, window=10),      # 161
    ])

print('Feature functions defined:')
print('  One-hot encoding   → 680 dims')
print('  3-mer frequency    →  64 dims')
print('  4-mer frequency    → 256 dims')
print('  Positional dinucs  → 272 dims')
print('  GC windows         → 161 dims')
print('  ──────────────────────────────')
print('  TOTAL              → 1,433 dims')

## Cell 5 — Build feature matrix (sample 20,000 for speed)

In [ ]:
## Cell 6 — Train models (ElasticNet per condition, with CV tuning)

from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# ElasticNet mixes L1 (Lasso) and L2 (Ridge) regularisation.
# alpha controls overall penalty strength; l1_ratio controls the mix:
#   l1_ratio=0 → pure Ridge,  l1_ratio=1 → pure Lasso
# ElasticNetCV finds the best alpha via cross-validation for each condition.
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9]   # grid searched by CV

models  = []
results = {}

print(f'{"Condition":<40}  {"α":>8}  {"l1":>4}  {"R²":>6}  {"RMSE":>6}  {"MAE":>6}  {"Pearson r":>9}  {"Spearman r":>10}')
print('-' * 108)

for i, cname in enumerate(COND_COLS):
    t0 = time.time()

    # Cross-validated ElasticNet: finds best alpha across 50-point log grid
    cv_model = ElasticNetCV(
        l1_ratio=L1_RATIOS,
        alphas=None,          # auto 50-point log grid
        cv=5,
        max_iter=2000,
        random_state=42,
        n_jobs=-1
    )
    cv_model.fit(X_train, Y_train[:, i])

    # Refit final model with the best hyperparameters found
    model = ElasticNet(
        alpha=cv_model.alpha_,
        l1_ratio=cv_model.l1_ratio_,
        max_iter=2000
    )
    model.fit(X_train, Y_train[:, i])
    y_pred = model.predict(X_test)
    y_true = Y_test[:, i]

    r2       = r2_score(y_true, y_pred)
    rmse     = np.sqrt(mean_squared_error(y_true, y_pred))
    mae      = mean_absolute_error(y_true, y_pred)
    pearson  = pearsonr(y_true, y_pred)[0]
    spearman = spearmanr(y_true, y_pred)[0]

    models.append(model)
    results[cname] = {
        'r2': float(r2), 'rmse': float(rmse), 'mae': float(mae),
        'pearson': float(pearson), 'spearman': float(spearman),
        'best_alpha': float(cv_model.alpha_),
        'best_l1_ratio': float(cv_model.l1_ratio_),
        'n_nonzero_coefs': int(np.sum(model.coef_ != 0))
    }
    print(f'{cname:<40}  {cv_model.alpha_:>8.4f}  {cv_model.l1_ratio_:>4.1f}  '
          f'{r2:>6.3f}  {rmse:>6.3f}  {mae:>6.3f}  {pearson:>9.3f}  {spearman:>10.3f}  '
          f'({time.time()-t0:.0f}s)')

print('\nAll ElasticNet models trained ✓')
print(f'\nNote: ElasticNet zeroes out irrelevant features (sparsity).')
for cname in COND_COLS:
    n = results[cname]['n_nonzero_coefs']
    print(f'  {cname:<40}: {n:,} / 1433 non-zero coefficients')

## Cell 6b — Comprehensive evaluation metrics & visualisations

In [ ]:
# ── Comprehensive evaluation metrics table ──────────────────────────────────
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Build tidy metrics DataFrame
metric_rows = []
for i, (cname, label) in enumerate(zip(COND_COLS, COND_LABELS)):
    r = results[cname]
    y_pred = models[i].predict(X_test)
    y_true = Y_test[:, i]
    # Mean Absolute Percentage Error (guard divide-by-zero)
    nonzero = y_true != 0
    mape = float(np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100)
    metric_rows.append({
        'Condition':      label,
        'R²':             round(r['r2'], 4),
        'RMSE':           round(r['rmse'], 4),
        'MAE':            round(r['mae'], 4),
        'MAPE (%)':       round(mape, 2),
        'Pearson r':      round(r['pearson'], 4),
        'Spearman r':     round(r['spearman'], 4),
        'Best α':         round(r['best_alpha'], 5),
        'Best l1_ratio':  round(r['best_l1_ratio'], 2),
        'Non-zero coefs': r['n_nonzero_coefs'],
    })

metrics_df = pd.DataFrame(metric_rows)

def color_r2(v):      return 'background-color: #c8e6c9' if v >= 0.4 else ('background-color: #fff9c4' if v >= 0.2 else 'background-color: #ffcdd2')
def color_pearson(v): return 'background-color: #c8e6c9' if v >= 0.6 else ('background-color: #fff9c4' if v >= 0.4 else 'background-color: #ffcdd2')
def color_rmse(v):    return 'background-color: #c8e6c9' if v <= 0.5 else ('background-color: #fff9c4' if v <= 1.0 else 'background-color: #ffcdd2')

styled_metrics = (
    metrics_df.style
    .applymap(color_r2,      subset=['R²'])
    .applymap(color_pearson, subset=['Pearson r', 'Spearman r'])
    .applymap(color_rmse,    subset=['RMSE', 'MAE'])
    .format({
        'R²': '{:.4f}', 'RMSE': '{:.4f}', 'MAE': '{:.4f}',
        'MAPE (%)': '{:.2f}', 'Pearson r': '{:.4f}', 'Spearman r': '{:.4f}',
        'Best α': '{:.5f}', 'Best l1_ratio': '{:.2f}', 'Non-zero coefs': '{:,}'
    })
    .set_caption('ElasticNet Evaluation Metrics — All 6 Conditions (green=good, yellow=fair, red=poor)')
)
display(styled_metrics)

# ── Radar / spider chart comparing conditions across 4 metrics ───────────────
METRICS = ['R²', 'Pearson r', 'Spearman r', '1 - norm(RMSE)']
rmse_vals = [results[c]['rmse'] for c in COND_COLS]
rmse_norm = np.array(rmse_vals) / max(rmse_vals)  # normalise 0-1

radar_data = np.array([
    [results[c]['r2'], results[c]['pearson'], results[c]['spearman'], 1 - rmse_norm[i]]
    for i, c in enumerate(COND_COLS)
])
radar_data = np.clip(radar_data, 0, 1)

N = len(METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close polygon

COLORS = ['#2196F3','#FF9800','#4CAF50','#9C27B0','#00BCD4','#F44336']

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(METRICS, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25','0.5','0.75','1.0'], fontsize=7, color='grey')
ax.set_title('ElasticNet Model Performance Radar\n(per condition)', fontsize=12, fontweight='bold', pad=20)

for vals, label, color in zip(radar_data, COND_LABELS, COLORS):
    v = vals.tolist() + vals[:1].tolist()
    ax.plot(angles, v, color=color, linewidth=2, label=label)
    ax.fill(angles, v, color=color, alpha=0.08)

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=8)
plt.tight_layout()
plt.savefig('metrics_radar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: metrics_radar.png')

# ── Feature sparsity comparison bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
n_nonzero = [results[c]['n_nonzero_coefs'] for c in COND_COLS]
bars = ax.barh(COND_LABELS, n_nonzero, color=COLORS, edgecolor='white', height=0.6)
ax.set_xlabel('Number of non-zero coefficients (out of 1,433)', fontsize=10)
ax.set_title('ElasticNet Sparsity — Features selected per condition', fontsize=11, fontweight='bold')
for bar, v in zip(bars, n_nonzero):
    ax.text(v + 5, bar.get_y() + bar.get_height()/2, f'{v:,}', va='center', fontsize=9)
ax.invert_yaxis()
ax.axvline(1433, color='gray', linestyle='--', linewidth=0.8, label='All 1,433 features')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('elasticnet_sparsity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: elasticnet_sparsity.png')

# Save extended metrics CSV
metrics_df.to_csv('extended_metrics.csv', index=False)
print(f'Saved: extended_metrics.csv')

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error

models  = []
results = {}

print(f'{"Condition":<40}  {"R²":>6}  {"RMSE":>6}')
print('-' * 58)
for i, cname in enumerate(COND_COLS):
    t0    = time.time()
    model = Ridge(alpha=1.0)
    model.fit(X_train, Y_train[:, i])
    y_pred = model.predict(X_test)
    r2   = r2_score(Y_test[:, i], y_pred)
    rmse = np.sqrt(mean_squared_error(Y_test[:, i], y_pred))
    models.append(model)
    results[cname] = {'r2': float(r2), 'rmse': float(rmse)}
    print(f'{cname:<40}  {r2:>6.3f}  {rmse:>6.3f}  ({time.time()-t0:.1f}s)')

print('\nAll models trained ✓')

## Cell 7 — Visualise model performance (R² bar chart)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

r2_vals  = [results[c]['r2']   for c in COND_COLS]
rmse_vals= [results[c]['rmse'] for c in COND_COLS]

colors = ['#2196F3','#FF9800','#4CAF50','#9C27B0','#00BCD4','#F44336']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance — Ridge Regression (17,000 training sequences)', fontsize=13, fontweight='bold')

# R² bars
ax = axes[0]
bars = ax.barh(COND_LABELS, r2_vals, color=colors, edgecolor='white', height=0.6)
ax.set_xlim(0, 0.6)
ax.set_xlabel('R² (test set)', fontsize=11)
ax.set_title('R² per condition', fontsize=11)
ax.axvline(0.4, color='gray', linestyle='--', linewidth=0.8, label='R²=0.40 reference')
ax.legend(fontsize=9)
for bar, v in zip(bars, r2_vals):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.3f}',
            va='center', fontsize=9)
ax.invert_yaxis()

# RMSE bars
ax = axes[1]
bars = ax.barh(COND_LABELS, rmse_vals, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('RMSE (log2 units)', fontsize=11)
ax.set_title('RMSE per condition', fontsize=11)
for bar, v in zip(bars, rmse_vals):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2, f'{v:.3f}',
            va='center', fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_performance.png')

## Cell 8 — In-silico mutagenesis (positional importance)

In [ ]:
def compute_positional_importance(seq, models, scaler, n_conditions=6):
    """
    For each of the 170 positions, mutate to all 3 alternatives.
    Mean |Δprediction| across alternatives = importance score.
    Returns: importance (SEQ_LEN × n_conditions), base_predictions (n_conditions,)
    """
    base_feats  = extract_features(seq).reshape(1, -1)
    base_scaled = scaler.transform(base_feats)
    base_preds  = np.array([m.predict(base_scaled)[0] for m in models])

    importance = np.zeros((SEQ_LEN, n_conditions), dtype=np.float32)
    for pos in range(SEQ_LEN):
        alts = [nt for nt in 'ATCG' if nt != seq[pos]]
        delta_sum = np.zeros(n_conditions)
        for alt in alts:
            mut_seq    = seq[:pos] + alt + seq[pos+1:]
            mut_scaled = scaler.transform(extract_features(mut_seq).reshape(1, -1))
            mut_preds  = np.array([m.predict(mut_scaled)[0] for m in models])
            delta_sum += np.abs(mut_preds - base_preds)
        importance[pos] = delta_sum / len(alts)

    return importance, base_preds


def find_active_regions(importance, threshold_pct=75):
    """Contiguous runs above threshold_pct importance → top-3 per condition."""
    regions_per_cond = []
    for c in range(importance.shape[1]):
        imp    = importance[:, c]
        thresh = np.percentile(imp, threshold_pct)
        active = imp > thresh
        regions, in_region, start = [], False, 0
        for i, a in enumerate(active):
            if a and not in_region:
                start, in_region = i, True
            elif not a and in_region:
                regions.append((start, i-1, float(imp[start:i].mean())))
                in_region = False
        if in_region:
            regions.append((start, len(active)-1, float(imp[start:].mean())))
        regions.sort(key=lambda x: -x[2])
        regions_per_cond.append(regions[:3])
    return regions_per_cond


# Analyse 5 promoters from the test set
N_GENES = 5
sample_genes = df_test.iloc[:N_GENES]['gene'].values
sample_seqs  = df_test.iloc[:N_GENES]['sequence'].values
sample_true  = Y_test[:N_GENES]

analysis_results = []
print(f'Running in-silico mutagenesis on {N_GENES} promoters ...')
for j, (gene, seq, true_vals) in enumerate(zip(sample_genes, sample_seqs, sample_true)):
    print(f'  [{j+1}/{N_GENES}] {gene} ...', end=' ', flush=True)
    t0 = time.time()
    importance, pred_vals = compute_positional_importance(seq, models, scaler)
    regions = find_active_regions(importance)
    analysis_results.append({
        'gene': gene,
        'sequence': seq,
        'predictions': {COND_COLS[i]: float(pred_vals[i]) for i in range(6)},
        'true_values': {COND_COLS[i]: float(true_vals[i]) for i in range(6)},
        'positional_importance': importance.tolist(),
        'active_regions': {COND_COLS[i]: regions[i] for i in range(6)},
    })
    print(f'done ({time.time()-t0:.1f}s)')

print('\nMutagenesis complete ✓')

## Cell 9 — Importance heatmap (6 conditions × 170 positions)

In [ ]:
import seaborn as sns

COLORS = ['#2196F3','#FF9800','#4CAF50','#9C27B0','#00BCD4','#F44336']
CMAPS  = ['Blues','Oranges','Greens','Purples','GnBu','Reds']

for res in analysis_results:
    gene = res['gene']
    imp  = np.array(res['positional_importance'])   # (170, 6)
    pred = res['predictions']

    fig, axes = plt.subplots(6, 1, figsize=(18, 7), sharex=True)
    fig.suptitle(f'{gene}  —  Per-position importance heatmap', fontsize=12, fontweight='bold', y=1.01)

    for c, (ax, cname, label, cmap) in enumerate(zip(axes, COND_COLS, COND_LABELS, CMAPS)):
        row = imp[:, c].reshape(1, -1)
        sns.heatmap(row, ax=ax, cmap=cmap, cbar=True,
                    cbar_kws={'shrink': 0.8, 'pad': 0.01},
                    xticklabels=False, yticklabels=False)
        val = pred[cname]
        color = 'green' if val >= 0 else 'red'
        ax.set_ylabel(label, fontsize=7.5, rotation=0, labelpad=145, va='center')
        ax.text(172, 0.5, f'{val:+.2f}', transform=ax.transData,
                fontsize=8, va='center', color=color, fontweight='bold')

    axes[-1].set_xlabel('Position in 170bp promoter sequence', fontsize=9)
    # Add tick marks at every 10bp
    axes[-1].set_xticks(range(0, 171, 10))
    axes[-1].set_xticklabels(range(1, 172, 10), fontsize=7, rotation=0)

    plt.tight_layout()
    fname = f'heatmap_{gene}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')

## Cell 10 — Annotated sequence view (colour-coded nucleotides + active regions)

In [ ]:
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import Normalize
from matplotlib import cm

NT_COLORS = {'A': '#e74c3c', 'T': '#3498db', 'C': '#2ecc71', 'G': '#f39c12'}

def plot_sequence(res, cond_idx=2):
    """
    Draw the 170bp sequence as coloured nucleotide tiles.
    Active positions (importance > 75th pct) are highlighted with a teal background.
    """
    gene      = res['gene']
    seq       = res['sequence']
    imp       = np.array(res['positional_importance'])[:, cond_idx]
    thresh    = np.percentile(imp, 75)
    cname     = COND_COLS[cond_idx]
    label     = COND_LABELS[cond_idx]
    pred_val  = res['predictions'][cname]

    cols_per_row = 50
    n_rows = -(-len(seq) // cols_per_row)   # ceiling division

    fig, ax = plt.subplots(figsize=(18, 1.2 * n_rows + 1))
    ax.set_xlim(-0.5, cols_per_row + 0.5)
    ax.set_ylim(-n_rows - 0.5, 0.8)
    ax.axis('off')
    ax.set_title(
        f'{gene}  |  Condition: {label}  |  Predicted: {pred_val:+.3f} log2',
        fontsize=10, fontweight='bold'
    )

    norm = Normalize(vmin=0, vmax=imp.max())

    for i, (nt, im) in enumerate(zip(seq, imp)):
        col = i % cols_per_row
        row = -(i // cols_per_row)
        is_active = im > thresh
        bg = '#d5f5e3' if is_active else '#f9f9f9'
        rect = FancyBboxPatch((col - 0.45, row - 0.45), 0.9, 0.9,
                              boxstyle='round,pad=0.05',
                              facecolor=bg, edgecolor='#cccccc', linewidth=0.4)
        ax.add_patch(rect)
        ax.text(col, row, nt, ha='center', va='center',
                fontsize=7, fontweight='bold',
                color=NT_COLORS.get(nt, 'black'),
                fontfamily='monospace')
        if (i + 1) % 10 == 0:
            ax.text(col, row - 0.52, str(i+1), ha='center', va='top',
                    fontsize=5.5, color='#888888')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d5f5e3', edgecolor='gray', label='Active (imp > 75th pct)'),
        Patch(facecolor='#f9f9f9', edgecolor='gray', label='Neutral'),
    ] + [Patch(facecolor=c, label=n) for n, c in NT_COLORS.items()]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=7,
              ncol=3, framealpha=0.8)

    plt.tight_layout()
    fname = f'seq_{gene}_cond{cond_idx}.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')


# Plot the first 2 promoters for condition 2 (with enhancer, dark, tobacco)
for res in analysis_results[:2]:
    plot_sequence(res, cond_idx=2)

## Cell 11 — Active regions summary table

In [ ]:
from IPython.display import display

rows = []
for res in analysis_results:
    gene = res['gene']
    for cond_idx, cname in enumerate(COND_COLS):
        pred = res['predictions'][cname]
        true = res['true_values'][cname]
        for (start, end, imp_score) in res['active_regions'][cname]:
            subseq = res['sequence'][start:end+1]
            rows.append({
                'Gene':        gene,
                'Condition':   COND_LABELS[cond_idx],
                'Start (1-idx)': start + 1,
                'End (1-idx)':   end + 1,
                'Length (bp)': end - start + 1,
                'Subsequence': subseq,
                'Importance':  round(imp_score, 4),
                'Predicted':   round(pred, 3),
                'True':        round(true, 3),
            })

summary_df = pd.DataFrame(rows).sort_values('Importance', ascending=False).reset_index(drop=True)

def color_pred(val):
    color = '#c8e6c9' if val >= 0 else '#ffcdd2'
    return f'background-color: {color}'

styled = (summary_df.style
          .applymap(color_pred, subset=['Predicted', 'True'])
          .format({'Importance': '{:.4f}', 'Predicted': '{:+.3f}', 'True': '{:+.3f}'})
          .set_caption('Active regions per promoter per condition (sorted by importance)'))

display(styled)

# Save CSV
summary_df.to_csv('active_regions.csv', index=False)
print(f'\nSaved active_regions.csv  ({len(summary_df)} rows)')

## Cell 12 — Predicted vs True scatter (all 6 conditions)

In [ ]:
COLORS = ['#2196F3','#FF9800','#4CAF50','#9C27B0','#00BCD4','#F44336']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()
fig.suptitle('Predicted vs True expression (test set, log2)', fontsize=13, fontweight='bold')

for i, (cname, label, color) in enumerate(zip(COND_COLS, COND_LABELS, COLORS)):
    ax    = axes[i]
    y_hat = models[i].predict(X_test)
    y_true= Y_test[:, i]
    r2    = results[cname]['r2']
    rmse  = results[cname]['rmse']

    ax.scatter(y_true, y_hat, alpha=0.25, s=4, color=color)
    lims = [min(y_true.min(), y_hat.min()) - 0.2,
            max(y_true.max(), y_hat.max()) + 0.2]
    ax.plot(lims, lims, 'k--', linewidth=0.8, alpha=0.5, label='ideal')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_title(f'{label}\nR²={r2:.3f}  RMSE={rmse:.3f}', fontsize=9)
    ax.set_xlabel('True (log2)', fontsize=8)
    ax.set_ylabel('Predicted (log2)', fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig('scatter_pred_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: scatter_pred_vs_true.png')

## Cell 13 — Predict a custom sequence

In [ ]:
# ─── CHANGE THIS TO ANY 170bp SEQUENCE YOU WANT TO TEST ──────────────────────
CUSTOM_SEQ = (
    'GAATATAACGAAAGTAGTACTTAATTTGTTTACATAATTTTATTTTTTTATTTTTACTTT'
    'GGAGAAATCAGTTAAGAGTAGGATTGATTATACATTTAATTTGCTTATTTAATTTTATTAGTTTT'
    'ATTAATTTAAAAAACATCGAATGACATATGTCATTTTGTTGTGTG'
)
CUSTOM_SEQ = CUSTOM_SEQ.upper().replace(' ', '')
# ─────────────────────────────────────────────────────────────────────────────

if len(CUSTOM_SEQ) != 170:
    print(f'⚠️  Sequence is {len(CUSTOM_SEQ)}bp — must be exactly 170bp.')
else:
    feats  = extract_features(CUSTOM_SEQ).reshape(1, -1)
    scaled = scaler.transform(feats)
    preds  = {cname: float(m.predict(scaled)[0]) for cname, m in zip(COND_COLS, models)}

    print('Predictions for custom sequence:')
    print(f'{"Condition":<40}  {"Predicted (log2)":>18}')
    print('-' * 62)
    for cname, label in zip(COND_COLS, COND_LABELS):
        v = preds[cname]
        bar  = '█' * int(abs(v) * 3)
        sign = '+' if v >= 0 else ''
        print(f'{label:<40}  {sign}{v:>6.3f}  {bar}')

    print('\nRunning positional importance ...')
    imp, _ = compute_positional_importance(CUSTOM_SEQ, models, scaler)
    regions = find_active_regions(imp)

    print('\nTop active regions (condition: with enhancer, dark, tobacco):')
    for start, end, score in regions[2]:
        subseq = CUSTOM_SEQ[start:end+1]
        print(f'  bp {start+1:3d}–{end+1:3d}  {subseq:<20}  importance={score:.4f}')

## Cell 14 — Save all outputs and download

In [ ]:
import pickle, json

# Save model bundle
with open('promoter_models.pkl', 'wb') as f:
    pickle.dump({'models': models, 'scaler': scaler,
                 'conditions': COND_COLS, 'performance': results}, f)

# Save analysis results
with open('analysis_results.json', 'w') as f:
    json.dump(analysis_results, f)

# Save performance table
perf_df = pd.DataFrame([
    {'Condition': COND_LABELS[i], 'R2': results[c]['r2'], 'RMSE': results[c]['rmse']}
    for i, c in enumerate(COND_COLS)
])
perf_df.to_csv('model_performance.csv', index=False)

print('Files saved:')
for f in ['promoter_models.pkl', 'analysis_results.json',
          'model_performance.csv', 'active_regions.csv',
          'model_performance.png', 'scatter_pred_vs_true.png']:
    import os
    if os.path.exists(f):
        print(f'  ✓  {f}  ({os.path.getsize(f)/1e3:.1f} KB)')

# Download everything
from google.colab import files
for fname in ['promoter_models.pkl', 'analysis_results.json',
              'model_performance.csv', 'active_regions.csv',
              'model_performance.png', 'scatter_pred_vs_true.png']:
    if os.path.exists(fname):
        files.download(fname)